# EMBER2024 Malware Detection — Results Analysis

This notebook visualises and compares:
- Our model (ResidualMLP + SupCon + prototypical) vs LightGBM baseline
- Detection PR curves and ROC curves by file type
- Family classification confusion matrix and t-SNE embedding plot
- Challenge set analysis

**Prerequisites**: Run `bash scripts/run_all.sh /path/to/ember2024` first, or `--small` mock data.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
    average_precision_score, roc_auc_score,
)
from sklearn.manifold import TSNE

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

matplotlib.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
})
sns.set_style('whitegrid')
COLORS = sns.color_palette('tab10')

RESULTS_DIR = Path('../results')
CKPT_DIR    = Path('../checkpoints')

## 1. Load Saved Results

In [ ]:
# Load our model results
our_results_path = RESULTS_DIR / 'results.json'
lgbm_results_path = RESULTS_DIR / 'lgbm_baseline_results.json'

our_results  = json.loads(our_results_path.read_text())  if our_results_path.exists()  else {}
lgbm_results = json.loads(lgbm_results_path.read_text()) if lgbm_results_path.exists() else {}

print(f'Our results keys:  {list(our_results.keys())[:10]}')
print(f'LGBM results keys: {list(lgbm_results.keys())[:10]}')

## 2. Detection Comparison Table

In [ ]:
import pandas as pd

FILE_TYPES = ['Overall', 'Win32PE', 'Win64PE', 'DotNetPE', 'APK', 'ELF', 'PDF']

rows = []
for ft in FILE_TYPES:
    key = f'detection/{ft}'
    our  = our_results.get(key, {})
    lgbm = lgbm_results.get('lgbm_detection_test', {}).get(ft, {})
    rows.append({
        'File Type':      ft,
        'Ours PR-AUC':    our.get('ours_pr_auc',  float('nan')),
        'LGBM PR-AUC':   lgbm.get('pr_auc',       float('nan')),
        'Δ PR-AUC':      our.get('delta_pr_auc',  float('nan')),
        'Ours ROC-AUC':  our.get('ours_roc_auc',  float('nan')),
        'LGBM ROC-AUC':  lgbm.get('roc_auc',      float('nan')),
    })

df = pd.DataFrame(rows).set_index('File Type')
df.style.format('{:.4f}').background_gradient(subset=['Δ PR-AUC'], cmap='RdYlGn')

## 3. Family Classification Table

In [ ]:
our_fam  = our_results.get('family/overall', {})
lgbm_fam = lgbm_results.get('lgbm_family_test', {})

fam_df = pd.DataFrame([
    {'Method': 'LightGBM Baseline',      'Accuracy': lgbm_fam.get('accuracy', float('nan')),  'Macro F1': lgbm_fam.get('macro_f1', float('nan'))},
    {'Method': 'Ours (softmax head)',     'Accuracy': our_fam.get('ours_accuracy', float('nan')),  'Macro F1': our_fam.get('ours_macro_f1', float('nan'))},
    {'Method': 'Ours (prototypical)',     'Accuracy': our_fam.get('proto_accuracy', float('nan')), 'Macro F1': our_fam.get('proto_macro_f1', float('nan'))},
]).set_index('Method')

print('Family Classification Results')
print('Paper baseline: accuracy=67.97%, macro F1=0.44')
print()
fam_df.style.format('{:.4f}')

## 4. PR Curves by File Type

In [ ]:
# To draw actual PR curves we need the raw scores.
# If you saved them during evaluate.py this loads them; otherwise shows the stored AUC values.

scores_path = RESULTS_DIR / 'raw_scores.npz'

if scores_path.exists():
    raw = np.load(scores_path)
    y_true   = raw['y_true']
    our_score = raw['our_score']
    lgbm_score = raw.get('lgbm_score', None)
    file_types = raw.get('file_types', None)

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, ft in zip(axes.flat, FILE_TYPES[1:]):
        if file_types is not None:
            mask = file_types == ft
        else:
            mask = np.ones(len(y_true), dtype=bool)

        if mask.sum() > 0 and len(np.unique(y_true[mask])) > 1:
            p, r, _ = precision_recall_curve(y_true[mask], our_score[mask])
            auc_val = average_precision_score(y_true[mask], our_score[mask])
            ax.plot(r, p, label=f'Ours (AUC={auc_val:.3f})', color=COLORS[0])

            if lgbm_score is not None:
                p2, r2, _ = precision_recall_curve(y_true[mask], lgbm_score[mask])
                auc2 = average_precision_score(y_true[mask], lgbm_score[mask])
                ax.plot(r2, p2, label=f'LightGBM (AUC={auc2:.3f})', color=COLORS[1], linestyle='--')

        ax.set_title(ft)
        ax.set_xlabel('Recall')
        ax.set_ylabel('Precision')
        ax.legend()
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

    plt.suptitle('PR Curves by File Type', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'pr_curves.pdf', bbox_inches='tight')
    plt.show()
else:
    print(f'Raw scores not found at {scores_path}.')
    print('Re-run evaluate.py with --save_raw_scores flag, or plot from stored AUC values above.')

## 5. t-SNE of Embeddings (Coloured by Family)

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from src.data_loader import load_ember_arrays, load_family_metadata, EMBER2024FamilyDataset
from src.model import build_contrastive_model
from src.utils import load_config, get_device

DATA_DIR = None  # Set this to your data directory
CKPT = CKPT_DIR / 'stage1' / 'best_model.pt'
N_TSNE_FAMILIES = 20   # Show top N families in t-SNE
N_TSNE_SAMPLES  = 50   # Samples per family for t-SNE

if DATA_DIR and CKPT.exists():
    cfg  = load_config('../configs/default.yaml')
    cfg['data']['data_dir'] = str(DATA_DIR)
    dev  = get_device('cuda')

    X_train, y_train = load_ember_arrays(str(DATA_DIR), 'train', use_memmap=True)
    meta = load_family_metadata(str(DATA_DIR), 'train')

    # Pick top-N families by sample count
    top_families = sorted(
        meta.family_to_label.keys(),
        key=lambda f: len(meta.family_to_indices.get(f, [])),
        reverse=True
    )[:N_TSNE_FAMILIES]

    top_labels = [meta.family_to_label[f] for f in top_families]
    label_set  = set(top_labels)

    # Sample indices
    sampled_idx  = []
    sampled_lbl  = []
    for f in top_families:
        idxs = meta.family_to_indices[f][:N_TSNE_SAMPLES]
        sampled_idx.extend(idxs)
        sampled_lbl.extend([meta.family_to_label[f]] * len(idxs))

    X_sub  = np.array(X_train[sampled_idx])
    labels_sub = np.array(sampled_lbl)

    # Extract embeddings
    model = build_contrastive_model(cfg).to(dev)
    ckpt  = torch.load(str(CKPT), map_location=dev)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    with torch.no_grad():
        embs = []
        for i in range(0, len(X_sub), 512):
            x = torch.from_numpy(X_sub[i:i+512]).to(dev)
            e = model.encode(x)
            e = F.normalize(e, dim=1)
            embs.append(e.cpu().numpy())
        embs = np.concatenate(embs, 0)

    print(f'Running t-SNE on {len(embs)} embeddings …')
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    coords = tsne.fit_transform(embs)

    fig, ax = plt.subplots(figsize=(12, 10))
    palette = sns.color_palette('tab20', N_TSNE_FAMILIES)
    for i, (f, lbl) in enumerate(zip(top_families, top_labels)):
        mask = labels_sub == lbl
        ax.scatter(coords[mask, 0], coords[mask, 1], c=[palette[i]], label=f, s=15, alpha=0.7)

    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax.set_title(f't-SNE of Contrastive Embeddings (Top {N_TSNE_FAMILIES} Families)', fontweight='bold')
    ax.set_xlabel('t-SNE dim 1')
    ax.set_ylabel('t-SNE dim 2')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'tsne_families.pdf', bbox_inches='tight')
    plt.show()
else:
    print('Set DATA_DIR and ensure checkpoints/stage1/best_model.pt exists to generate t-SNE.')

## 6. Challenge Set Analysis

In [ ]:
challenge = our_results.get('challenge_set', {})

print('Challenge Set (6,315 evasive malware)')
print(f"  Samples:          {challenge.get('n_samples', 'N/A'):,}")
print(f"  Detection rate:   {challenge.get('detection_rate', float('nan')):.2%}")
print(f"  Avg det. score:   {challenge.get('avg_detection_score', float('nan')):.4f}")
print(f"  Family accuracy:  {challenge.get('family', {}).get('accuracy', float('nan')):.4f}")
print(f"  Family macro F1:  {challenge.get('family', {}).get('macro_f1', float('nan')):.4f}")

## 7. Training Loss Curves (if W&B logs available)

In [ ]:
# If you used wandb, pull run history here.
# Otherwise, we plot dummy curves for illustration.

USE_WANDB = False  # Set True and fill in run_path if you have W&B logs

if USE_WANDB:
    import wandb
    api = wandb.Api()
    run = api.run('your-entity/ember2024/run_id')  # replace with your run path
    history = run.history()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    history['train/loss'].dropna().plot(ax=axes[0], label='Train')
    history['val/loss'].dropna().plot(ax=axes[0], label='Val')
    axes[0].set_title('Stage 1: SupCon Loss'); axes[0].legend()
    history.get('val/det_pr_auc', history['val/loss']).dropna().plot(ax=axes[1])
    axes[1].set_title('Stage 2: Val PR-AUC')
    plt.tight_layout(); plt.show()
else:
    print('Set USE_WANDB=True and provide your W&B run path for training curves.')

## 8. Prototype Distance Distribution

In [ ]:
proto_path = CKPT_DIR / 'stage3' / 'prototypes.npz'

if proto_path.exists():
    saved = np.load(proto_path, allow_pickle=True)
    protos = saved['prototypes']   # (C, D)

    # Inter-prototype cosine similarities
    sims = protos @ protos.T     # (C, C)
    np.fill_diagonal(sims, np.nan)
    inter_sims = sims[~np.isnan(sims)]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(inter_sims, bins=50, color=COLORS[0], edgecolor='white')
    axes[0].set_title('Inter-Prototype Cosine Similarity Distribution')
    axes[0].set_xlabel('Cosine Similarity')
    axes[0].set_ylabel('Count')
    axes[0].axvline(np.nanmedian(inter_sims), color='red', linestyle='--',
                    label=f'Median: {np.nanmedian(inter_sims):.3f}')
    axes[0].legend()

    # L2 norm of each prototype (should be ~1 for well-formed protos)
    norms = np.linalg.norm(protos, axis=1)
    axes[1].hist(norms, bins=30, color=COLORS[1], edgecolor='white')
    axes[1].set_title('Prototype L2 Norm Distribution')
    axes[1].set_xlabel('L2 Norm')

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'prototype_distances.pdf', bbox_inches='tight')
    plt.show()
    print(f'Prototypes loaded: {protos.shape[0]} classes, {protos.shape[1]} dims')
    print(f'Median inter-prototype sim: {np.nanmedian(inter_sims):.4f}')
else:
    print(f'Prototypes not found at {proto_path}. Run Stage 3 first.')

## 9. Publication Summary Table

In [ ]:
# Clean comparison table suitable for a report/paper
summary = {
    'Task':            ['Detection (Overall)', 'Detection (Win32PE)', 'Detection (ELF)',
                        'Family Classification', 'Challenge Set Detection'],
    'Metric':          ['PR-AUC', 'PR-AUC', 'PR-AUC', 'Macro F1', 'Detection Rate'],
    'LightGBM (paper baseline)': [
        lgbm_results.get('lgbm_detection_test', {}).get('Overall', {}).get('pr_auc', '—'),
        lgbm_results.get('lgbm_detection_test', {}).get('Win32PE', {}).get('pr_auc', '—'),
        lgbm_results.get('lgbm_detection_test', {}).get('ELF',     {}).get('pr_auc', '—'),
        lgbm_results.get('lgbm_family_test',    {}).get('macro_f1', '—'),
        '—',
    ],
    'Ours': [
        our_results.get('detection/Overall',  {}).get('ours_pr_auc', '—'),
        our_results.get('detection/Win32PE',  {}).get('ours_pr_auc', '—'),
        our_results.get('detection/ELF',      {}).get('ours_pr_auc', '—'),
        our_results.get('family/overall',     {}).get('ours_macro_f1', '—'),
        our_results.get('challenge_set',      {}).get('detection_rate', '—'),
    ],
}

pd.DataFrame(summary).set_index('Task').style.format(
    lambda x: f'{x:.4f}' if isinstance(x, float) else x
)